In [5]:
import requests
import pandas as pd
import io
import json

# ==============================================================================
# CONFIGURATION FOR MULTIPLE DATASETS (THE LIST)
# ==============================================================================
# This is a Python list (enclosed in [ ]). 
# You can add as many dictionaries { ... } inside this list as you want.
# Each dictionary represents a different dataset to download.

TARGET_DATASETS = [
    
    # --- DATASET 1: R&D Expenditure ---
    {
        "file_name": "OECD_RD_GDP_2000_2025.csv",
        "agency_id": "OECD.STI.STP",
        "dataflow_id": "DSD_MSTI@DF_MSTI",
        "filters": {
            # We keep UNIT_MEASURE because your log confirmed 'PT_B1GQ' exists.
            # We temporarily remove 'MEASURE' so the script downloads the data,
            # allowing you to open the CSV and choose the correct MEASURE code.
            "UNIT_MEASURE": "PT_B1GQ" 
        }
    },
    
    # --- DATASET 2: Placeholder for Public Debt (Example) ---
    # {
    #     "file_name": "OECD_Public_Debt.csv",
    #     "agency_id": "INSERT_AGENCY_ID",
    #     "dataflow_id": "INSERT_DATAFLOW_ID",
    #     "filters": {
    #         "UNIT_MEASURE": "INSERT_CODE"
    #     }
    # },

    # --- DATASET 3: Placeholder for Inflation (Example) ---
    # {
    #     "file_name": "OECD_Inflation.csv",
    #     "agency_id": "INSERT_AGENCY_ID",
    #     "dataflow_id": "INSERT_DATAFLOW_ID",
    #     "filters": {
    #         "MEASURE": "INSERT_CODE"
    #     }
    # }
]

# Time filters and API constants
START_PERIOD = "2000"
END_PERIOD = "2025"
VERSION = "all"
DIMENSIONS = "all"

headers = {
    "Accept": "application/vnd.sdmx.data+csv; charset=utf-8; version=2",
    "User-Agent": "DataExtractionScript/2.0 (Python/Requests)"
}

# ==============================================================================
# BATCH EXTRACTION PROCESS
# ==============================================================================

for dataset in TARGET_DATASETS:
    print(f"Processing: {dataset['file_name']}...")
    
    url = f"https://sdmx.oecd.org/public/rest/data/{dataset['agency_id']},{dataset['dataflow_id']},{VERSION}/{DIMENSIONS}"
    
    params = {
        "startPeriod": START_PERIOD,
        "endPeriod": END_PERIOD,
        "dimensionAtObservation": "AllDimensions",
        "format": "csvfilewithlabels"
    }
    
    response = requests.get(url, params=params, headers=headers)
    
    if response.status_code == 200:
        csv_stream = io.StringIO(response.text)
        df = pd.read_csv(csv_stream, low_memory=False)
        df_clean = df.dropna(subset=['OBS_VALUE']).copy()
        
        # Apply the specific filters requested
        for column_name, filter_value in dataset.get('filters', {}).items():
            if column_name in df_clean.columns:
                df_clean = df_clean[df_clean[column_name] == filter_value]
            else:
                print(f"  -> Warning: Column '{column_name}' not found.")
        
        # Failsafe: if dataframe is empty after filtering, export all available codes for debugging
        if df_clean.empty:
            print(f"  -> Error: Dataset is empty after applying filters.")
            
            debug_dict = {}
            for col in df.columns:
                if col not in ['OBS_VALUE', 'TIME_PERIOD', 'REF_AREA']:
                    debug_dict[col] = df[col].dropna().unique().tolist()
            
            with open("DEBUG_OECD_CODES.txt", "w") as f:
                f.write(json.dumps(debug_dict, indent=4))
                
            print("  -> I saved a file named 'DEBUG_OECD_CODES.txt' in this folder.")
            print("  -> Open it to see ALL the exact codes you can use in your filters.")
            continue
        
        # Keep only the requested columns
        target_columns = ['REF_AREA', 'TIME_PERIOD', 'OBS_VALUE']
        
        if all(col in df_clean.columns for col in target_columns):
            df_tidy = df_clean[target_columns].rename(columns={
                'REF_AREA': 'Country',
                'TIME_PERIOD': 'Year',
                'OBS_VALUE': 'Value'
            })
            
            df_tidy = df_tidy.reset_index(drop=True)
            df_tidy.to_csv(dataset['file_name'], index=False)
            
            print(f"  -> Success! Saved {len(df_tidy)} rows to {dataset['file_name']}.\n")
        else:
            print(f"  -> Structural error: Missing SDMX columns in {dataset['file_name']}.\n")
    else:
        print(f"  -> HTTP Error {response.status_code} for {dataset['file_name']}.\n")

Processing: OECD_RD_GDP_2000_2025.csv...
  -> Success! Saved 9812 rows to OECD_RD_GDP_2000_2025.csv.

